# CR Algorithm Test

## Useful Functions

In [43]:
import numpy as np

from pathlib import Path
import sys
parent_dir = Path.cwd().parent.resolve()
sys.path.append(str(parent_dir))

np.set_printoptions(precision=3)

In [44]:
from numpy.linalg import norm
from numpy import sin, cos, tan, sqrt, pi
import matplotlib.pyplot as plt
from datetime import datetime
from dataclasses import dataclass
import pytz

In [45]:
mu_unicode = "\u03bc"
deg_unicode = "\u00b0"

In [46]:
def circle_points(radius, N = 100, position = None):
    p = (0,0) if position is None else position
    t = np.linspace(0, pi * 2, N, endpoint=False)
    return [radius * cos(t) + p[0], radius * sin(t) + p[1]]

def noise(cov):
    return np.random.multivariate_normal(
        mean = np.zeros(cov.shape[0]),
        cov=cov
    )

R_example = np.array([
    [0.5, 0, 0],
    [0, 0.5, 0],
    [0, 0, 0]
])
# print(f"Example noise: {noise(R_example)}\nwith R = \n{R_example}")

## Setup

In [47]:
import trajectory as T
from PlanetaryData import Earth, Luna, Sun
from Spice import SpiceImporter
from algorithms import ChristianRobinson
from Constants import RAD_TO_DEG, DEG_TO_RAD, ARCSEC_TO_RAD, RAD_TO_ARCSEC

## Camera

In [48]:
# # Camera specs 
# # e.g. https://spinworks.pt/products/star-tracker-st1/

# # Inputs
# fov_diag = 18.2 * DEG_TO_RAD
# f = 50e-3               # focal length
# n_pixels = (2048, 2048)
# mu = 5.5e-6             # pixel spacing [m]
# sigma = 2 * ARCSEC_TO_RAD / 3 # Gives in 3-sigma

# # Calculated
# mu_angle = mu / f       # pixel angle resolution [rad]
# print(f"Pixel anglular resolution: {mu_angle*RAD_TO_ARCSEC:.2f} arcseconds")
# fov = fov_diag / np.sqrt(2)
# print(f"FOV: {fov*RAD_TO_DEG:.2f}º")

In [49]:
# Camera specs 
# e.g. https://satsearch.co/products/arcsec-sagitta-star-tracker?utm_source=chatgpt.com

camera_model = "sagitta"

if camera_model == "sagitta":
    # Inputs
    fov = 25.2 * DEG_TO_RAD
    f = 2500e-3               # focal length
    n_pixels = np.array([2048, 2048])   # Assumed from previous star tracker

    cross_boresight_sigma = 2 * ARCSEC_TO_RAD
    boresight_sigma = 10 * ARCSEC_TO_RAD 
    # Calculated
    mu_angle = fov/n_pixels[0]       # pixel angle resolution [rad/px]
    mu = f * mu_angle             # pixel spacing [m/px]

    
print(f"Pixel anglular resolution: {mu_angle*RAD_TO_ARCSEC:.2f} arcs/px")
print(f"Pixel resolution: {mu*1000000:.2f} {mu_unicode}m/px")
print(f"FOV: {fov*RAD_TO_DEG:.2f}{deg_unicode}")

Pixel anglular resolution: 44.30 arcs/px
Pixel resolution: 536.89 μm/px
FOV: 25.20°


In [50]:
# @dataclass
# class CameraSpecs:
#     f: float
#     mu: float
#     n_pixels: float
#     alpha: float = 0
#     # u_p: float
#     # v_p: float
#     # dx: float
#     # dy: float

# @dataclass
# class WorldSpecs:
#     r_truth: np.ndarray
#     a: float
#     b: float
#     c: float

# @dataclass
# class ImageSpecs:
#     radial_points: float
#     sigma_cross_boresight: float
#     planet_center_loc_px: float

# class Image:
#     R: np.ndarray
#     u_noise: np.ndarray
#     u_truth: np.ndarray
    


In [51]:
class Camera:
    def __init__(self,
        f: float,
        mu: float,
        resolution: float
    ):
        """Camera object

        Args:
            f (float): Focal length [m]
            mu (float): Pixel resolution [m/px]
            resolution (np.ndarray(2)): Total pixels (x, y) [px]
        """
        self.f = f
        self.mu = mu
        self.resolution = resolution

    def calc_specs(self, alpha_skew: float = 0,):

        dx = dy = f / mu
        u_p, v_p = self.resolution / 2

        self.K_inv = np.array([
            [1/dx, -alpha_skew/dx/dy, (alpha_skew*v_p - dy*u_p)/dx/dy],
            [0, 1/dy, -v_p/dy],
            [0, 0, 1]
        ])

        self.u_p = u_p
        self.v_p = v_p
        self.dx = dx
        self.dy = dy

In [52]:
class WorldSpecs:
    def __init__(self,
        r_truth: np.ndarray,
        a: float,
        b: float,
        c: float
    ):
        """

        Args:
            r_truth (np.ndarray): Position relative to central body [m]
            a (float): Ellipsoid parameter [m]
            b (float): Ellipsoid parameter [m]
            c (float): Ellipsoid parameter [m]
        """
        self.r_truth = r_truth
        self.a = a
        self.b = b
        self.c = c

In [53]:
class PlanetImage:
    def __init__(self, c: Camera, w: WorldSpecs, N_PLANET_POINTS: int, sigma_cross_boresight_px: float = 0.5,
                 planet_center_px: np.ndarray = None, planet_radius_px: float = None):
        """Init and generate planet points

        Args:
            c (Camera): Camera
            w (WorldSpecs): World specs with r_truth and ellipsoid parameters
            N_PLANET_POINTS (int): Number of planet points to generate 
            sigma_cross_boresight_px (float, optional): Standard deviation [px]. Defaults to 0.5.
            planet_center_px (np.ndarray(2), optional): Planet center [px]. Defaults to image center.
            planet_radius_px (float, optional): Planet radius [px]. Only if you want to manually specify.
        """

        # Planet pixel locations in image
        planet_center_px = (
            np.array([c.u_p, c.v_p]) if planet_center_px is None else planet_center_px
        )
        planet_angle = np.arcsin(w.a / norm(w.r_truth))
        planet_radius_px = (
            c.dx * tan(planet_angle) if planet_radius_px is None else planet_radius_px
        )

        # Generate points
        self.N = N_PLANET_POINTS
        self.body_points = circle_points(planet_radius_px, self.N,
                                    position=planet_center_px)
        self.u_points = np.vstack((self.body_points, np.ones(N_PLANET_POINTS))).T 

        # Noise
        s = sigma_cross_boresight_px
        self.R = np.array([
            [s**2,0,0],
            [0,s**2,0],
            [0,0,0]
        ])
        self.u_noise = np.array([v + noise(self.R) for v in self.u_points])

In [54]:
camera = Camera(f, mu, n_pixels)
camera.calc_specs()

world = WorldSpecs(
    np.array([0, 0, 7000e3]),
    a=3000e3,
    b=3000e3,
    c=3000e3
)

img = PlanetImage(camera, world, 100, 0.5)

cr_alg = ChristianRobinson(camera.K_inv, world.a, world.b, world.c)

In [55]:


T_p_c = np.eye(3)
sc_rot_angle = 90 * DEG_TO_RAD
T_p_c = np.array([
    [1, 0, 0],
    [0, cos(sc_rot_angle), sin(sc_rot_angle)],
    [0, -sin(sc_rot_angle), cos(sc_rot_angle)]
])

rc = cr_alg.run(img.u_noise, T_p_c)

print(f"CR Position: {rc/1000} km")
print("CR stats:")
print(f"- Absolute error: {(rc - world.r_truth)} km")
print(f"- Relative error: {abs(rc - world.r_truth)/norm(world.r_truth)}")
# 

CR Position: [-2.006e-02  1.700e-01  7.000e+03] km
CR stats:
- Absolute error: [-20.056 169.969  96.242] km
- Relative error: [2.865e-06 2.428e-05 1.375e-05]
